# Phase 3D — Memory Diagnostic Case Study

An **interventional diagnostic** that decomposes where the autoresearch agent's memory fails,
using genuine LLM decisions (not constructed labels). This applies the 3-probe framework from
Phase 2 to the autoresearch traces from Phase 3.

**Attribution:** adapted from *Agent Memory Techniques* by Nir Diamant (Apache-2.0) for the
taxonomy; the diagnostic methodology and inline code are original to this tutorial.

## The methodology

**Unit of analysis:** a *decision point* — a frozen pre-decision snapshot (history + candidates).

For each OOM config and each possible pre-decision history:
1. **Freeze** the state (never modified during probing — no temporal leakage).
2. Run **5 memory conditions** × K=5 repeats, all from the SAME frozen state.
3. **Metric:** proposal rate for the OOM config (fraction of K calls that proposed it).

| Condition | What the agent sees |
|---|---|
| M0 no-memory | Empty history (placebo baseline) |
| M1 raw-history | Full trial log (upstream autoresearch style) |
| M2 retrieved | Only same-batch trials (simulates batch-keyed retrieval) |
| M3 structured-rule | Raw history + inferred constraint ('depth>=D at batch B -> OOM') |
| M4 oracle | Raw history + actual outcome ('this config WILL OOM') |

## The results (9 decision points × 5 conditions × 5 repeats)

| Condition | Mean OOM-config proposal rate | Mean any-OOM rate |
|---|---|---|
| M0 no-memory | **50%** | 62% |
| M1 raw-history | **12%** | 12% |
| M2 retrieved (same-batch) | 12% | **30%** |
| M3 structured-rule | **5%** | 5% |
| M4 oracle | **0%** | 12% |

**Key comparisons:**
- **Information effect (M0→M1): -38pp.** Any trial history cuts OOM proposals by 76%.
- **Retrieval filtering (M1→M2): any-OOM INCREASES 12%→30%.** Same-batch-only retrieval is
 counterproductive — the agent loses cross-batch context and proposes MORE OOMs at other batches.
- **Structured rules (M1→M3): -7pp.** Explicit constraints help modestly.
- **Constraint quality (M3→M4): -5pp.** The inferred constraint nearly matches the oracle.

**VRAM model:** FP=7%, FN=46% (unreliable — misses ~half the OOMs; raw history outperforms it).

## What this teaches

1. **Memory representation matters.** Raw history (M1) dramatically outperforms no memory (M0).
2. **Naive retrieval filtering can HURT.** Filtering to same-batch trials (M2) causes the agent
 to lose cross-task context → more OOMs. This is non-obvious and has practical implications
 for how memory systems should retrieve experiment histories.
3. **Structured constraints approach oracle performance.** Deriving 'depth>=D at batch B -> OOM'
 from prior failures (M3) reduces OOM proposals to 5%, near the oracle's 0%.
4. **Simple VRAM extrapolation is unreliable** (46% false-negative). The agent's qualitative
 reasoning from raw history is more effective than a linear resource model.

**Connection to Phase 2:** this IS the 3-probe diagnostic framework (relevance/utilization/failure)
applied to autoresearch memory — measuring whether prior experiment records are *retrieved*,
*represented in an actionable form*, and *actually used* by the agent.

## Limitations (honest)

This is a **pilot case study**, not a validated benchmark:
- One frozen surface (10 configs, 3 unique OOMs).
- One LLM model (gpt-4o).
- ~7-9 unique decision states (small N).
- No real retriever (M2 simulates batch-keyed retrieval, not BM25/dense).
- K=5 repeats per condition (limited statistical power).

The finding ('retrieval filtering hurts') is a **hypothesis generator** from one surface/model.
A full study would need 30-50 decision states across multiple surfaces and models.
See `review-stage/AUTO_REVIEW.md` for the full adversarial review history (7 rounds).

In [ ]:
# Re-run the diagnostic (needs an OpenAI-compatible endpoint + the frozen surface)
# results = !python ../../scripts/phase3_diagnostic_v3.py
# Or inspect the pre-computed results:
import json, pathlib
results_path = pathlib.Path('../../results/phase3_diagnostic_v3.json')
if results_path.exists():
 data = json.loads(results_path.read_text())
 print(f"model: {data['model']}")
 print(f"decision points: {data['n_points']}, K={data['K']}")
 for r in data['results'][:5]:
 print(f" target={r['target']} {r['condition']:20s} -> target_rate={r['target_proposal_rate']}")
else:
 print('Run python scripts/phase3_diagnostic_v3.py first (needs OPENAI_API_KEY)')
